<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

Aquí hago la parte de preparar documentos: cargo los textos de Gutenberg que distribuye NLTK, cuento los documentos y sus tamaños, y aplico la normalización definida arriba. Los tamaños se reportan en caracteres y tokens.

In [4]:
import re
import sys
import time
import math
import statistics
from collections import Counter, defaultdict

import nltk
from nltk.corpus import gutenberg
import pandas as pd

try:
    FILE_IDS = gutenberg.fileids()
except LookupError:
    nltk.download('gutenberg')
    FILE_IDS = gutenberg.fileids()

assert len(FILE_IDS) == 18, f"Se esperaban 18 textos de NLTK; se encontraron {len(FILE_IDS)}"

TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", flags=re.UNICODE)

def tokenize(text):
    'Normaliza a minúsculas y conserva palabras Unicode y apóstrofos internos.'
    return TOKEN_RE.findall(text.lower().replace('\u2019', "'"))

documents = {file_id: tokenize(gutenberg.raw(file_id)) for file_id in FILE_IDS}
doc_ids = list(documents)
corpus_stats = pd.DataFrame([
    {"documento": doc_id, "caracteres": len(gutenberg.raw(doc_id)),
     "tokens": len(documents[doc_id])}
    for doc_id in doc_ids
])
print(f"Documentos: {len(documents)} | Tokens totales: {sum(map(len, documents.values())):,}")
display(corpus_stats)
print("Preprocesamiento: minúsculas; puntuación separa/elimina tokens; sin stopwords ni stemming.")


Documentos: 18 | Tokens totales: 2,182,064


,documento,caracteres,tokens
0,austen-emma.txt,887071,160999
1,austen-persuasion.txt,466292,83660
2,austen-sense.txt,673022,119965
3,bible-kjv.txt,4332554,852256
4,blake-poems.txt,38153,6839
5,bryant-stories.txt,249439,46321
6,burgess-busterbrown.txt,84663,15969
7,carroll-alice.txt,144395,26688
8,chesterton-ball.txt,457450,82248
9,chesterton-brown.txt,406629,72391


Preprocesamiento: minúsculas; puntuación separa/elimina tokens; sin stopwords ni stemming.


**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [8]:
# Recorro documentos y tokens con los dos lazos requeridos para búsqueda secuencial.
def sequential_search(term, corpus=documents):
    term_tokens = tokenize(term)
    if len(term_tokens) != 1:
        raise ValueError("La consulta secuencial debe contener exactamente un término")
    query = term_tokens[0]
    found = []
    for doc_id, tokens in corpus.items():
        contains_term = False
        for token in tokens:
            if token == query:
                contains_term = True
                break
        if contains_term:
            found.append(doc_id)
    return found

document_frequency = Counter(token for tokens in documents.values() for token in set(tokens))

# Tomo los términos de ejemplo de las frecuencias reales del corpus.
frequent_term = document_frequency.most_common(1)[0][0]
rare_term = sorted(term for term, df in document_frequency.items() if df == 1)[0]
missing_term = "zzqterminoinexistente"
sample_queries = {"frecuente": frequent_term, "poco frecuente": rare_term,
                  "inexistente": missing_term}
for label, term in sample_queries.items():
    print(f"{label}: {term!r} -> {sequential_search(term)}")


frecuente: 'place' -> ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
poco frecuente: '0' -> ['whitman-leaves.txt']
inexistente: 'zzqterminoinexistente' -> []


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [11]:
# Mido cada consulta 30 veces y convierto nanosegundos a milisegundos.
REPEATS = 30

def mean_query_ms(function, *args, repeats=REPEATS, **kwargs):
    durations = []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        function(*args, **kwargs)
        durations.append(time.perf_counter_ns() - start)
    # El promedio reduce el efecto de pequeñas variaciones entre ejecuciones.
    return statistics.mean(durations) / 1_000_000

sequential_times = pd.DataFrame([
    {"tipo": label, "término": term,
     "documentos_recuperados": len(sequential_search(term)),
     "media_30_ms": mean_query_ms(sequential_search, term)}
    for label, term in sample_queries.items()
])
display(sequential_times)

,tipo,término,documentos_recuperados,media_30_ms
0,frecuente,place,18,1.310850
1,poco frecuente,0,1,50.796173
2,inexistente,zzqterminoinexistente,0,50.882690


**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [21]:
# Construyo un índice invertido: cada término apunta a los documentos que lo contienen.
def build_inverted_index(corpus):
    index = defaultdict(set)
    for doc_id, tokens in corpus.items():
         # El set evita procesar varias veces un término repetido en el mismo documento.
        for token in set(tokens):
            index[token].add(doc_id)
    return dict(index)

inverted_index = build_inverted_index(documents)
all_doc_ids = set(documents)

def postings(term, index=inverted_index):
    normalized = tokenize(term)
    if len(normalized) != 1:
        raise ValueError("Se requiere exactamente un término")
    return set(index.get(normalized[0], set()))

def boolean_and(term_a, term_b, index=inverted_index):
    return postings(term_a, index) & postings(term_b, index)

def boolean_or(term_a, term_b, index=inverted_index):
    return postings(term_a, index) | postings(term_b, index)

def boolean_not(term, index=inverted_index, universe=all_doc_ids):
    # NOT devuelve todos los documentos del corpus que no contienen el término.
    return set(universe) - postings(term, index)

terms_to_inspect = [frequent_term, rare_term, "the"]
display(pd.DataFrame([
    {"término": term, "postings": sorted(postings(term)), "DF": len(postings(term))}
    for term in dict.fromkeys(terms_to_inspect)
]))
# Elijo otro término frecuente para mostrar ejemplos de AND y OR.
second_term = next((term for term, _ in document_frequency.most_common(2)
                    if term != frequent_term), frequent_term)
print("AND:", sorted(boolean_and(frequent_term, second_term)))
print("OR:", sorted(boolean_or(frequent_term, second_term)))
print("NOT (Complemento en los 18 documentos):", sorted(boolean_not(frequent_term)))


,término,postings,DF
0,place,"[austen-emma.txt, austen-persuasion.txt, auste...",18
1,0,[whitman-leaves.txt],1
2,the,"[austen-emma.txt, austen-persuasion.txt, auste...",18


AND: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
OR: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
NOT (Complemento en los 18 documentos): []


**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [26]:
# Comparo los resultados exactos y los tiempos de los dos métodos.
comparison_rows = []
for label, term in sample_queries.items():
    seq_result = set(sequential_search(term))
    idx_result = postings(term)
    # Si falla la igualdad, se detiene la comparación y se identifica la consulta.
    assert seq_result == idx_result, f"Diferencia para {term!r}"
    # Guardo una fila por método: consulta, cantidad de documentos y tiempo promedio.
    comparison_rows.extend([
        {"consulta": label, "término": term, "método": "secuencial",
         "documentos": len(seq_result), "media_30_ms": mean_query_ms(sequential_search, term)},
        {"consulta": label, "término": term, "método": "índice invertido",
         "documentos": len(idx_result), "media_30_ms": mean_query_ms(postings, term)},
    ])
comparison_small = pd.DataFrame(comparison_rows)
display(comparison_small)
print("Verificación: En ambos métodos devuelven los mismos documentos en las tres consultas que se hicieron.")

,consulta,término,método,documentos,media_30_ms
0,frecuente,place,secuencial,18,0.780957
1,frecuente,place,índice invertido,18,0.001347
2,poco frecuente,0,secuencial,1,50.330947
3,poco frecuente,0,índice invertido,1,0.001313
4,inexistente,zzqterminoinexistente,secuencial,0,52.329513
5,inexistente,zzqterminoinexistente,índice invertido,0,0.003523


Verificación: En ambos métodos devuelven los mismos documentos en las tres consultas que se hicieron.


### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.